In [72]:
import math
import nltk
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer
import pandas as pd
import pickle

In [37]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Gowtham\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

# RecipeNLG Dataset

### Create the sample dataset

In [ ]:
# Read the dataset
df = pd.read_csv("dataset/full_dataset.csv")
print(df.head()), print(len(df))

   Unnamed: 0                  title  \
0           0    No-Bake Nut Cookies   
1           1  Jewell Ball'S Chicken   
2           2            Creamy Corn   
3           3          Chicken Funny   
4           4   Reeses Cups(Candy)     

                                         ingredients  \
0  ["1 c. firmly packed brown sugar", "1/2 c. eva...   
1  ["1 small jar chipped beef, cut up", "4 boned ...   
2  ["2 (16 oz.) pkg. frozen corn", "1 (8 oz.) pkg...   
3  ["1 large whole chicken", "2 (10 1/2 oz.) cans...   
4  ["1 c. peanut butter", "3/4 c. graham cracker ...   

                                          directions  \
0  ["In a heavy 2-quart saucepan, mix brown sugar...   
1  ["Place chipped beef on bottom of baking dish....   
2  ["In a slow cooker, combine all ingredients. C...   
3  ["Boil and debone chicken.", "Put bite size pi...   
4  ["Combine first four ingredients and press in ...   

                                              link    source  \
0   www.cookbooks.com

In [ ]:
# Get the columns
df.columns

Index(['Unnamed: 0', 'title', 'ingredients', 'directions', 'link', 'source',
       'NER'],
      dtype='object')

In [ ]:
# Drop the Unnamed: 0 column
df.drop(columns=["Unnamed: 0"], inplace=True)
print(df.head())

                   title                                        ingredients  \
0    No-Bake Nut Cookies  ["1 c. firmly packed brown sugar", "1/2 c. eva...   
1  Jewell Ball'S Chicken  ["1 small jar chipped beef, cut up", "4 boned ...   
2            Creamy Corn  ["2 (16 oz.) pkg. frozen corn", "1 (8 oz.) pkg...   
3          Chicken Funny  ["1 large whole chicken", "2 (10 1/2 oz.) cans...   
4   Reeses Cups(Candy)    ["1 c. peanut butter", "3/4 c. graham cracker ...   

                                          directions  \
0  ["In a heavy 2-quart saucepan, mix brown sugar...   
1  ["Place chipped beef on bottom of baking dish....   
2  ["In a slow cooker, combine all ingredients. C...   
3  ["Boil and debone chicken.", "Put bite size pi...   
4  ["Combine first four ingredients and press in ...   

                                              link    source  \
0   www.cookbooks.com/Recipe-Details.aspx?id=44874  Gathered   
1  www.cookbooks.com/Recipe-Details.aspx?id=699419  Gathered

In [ ]:
# Get a sample of 5000 recipes
sample_df = df.sample(n=5000, random_state=42)
print(len(sample_df))
sample_df.to_csv("dataset/sample_dataset.csv", index=False)

5000


### Prepare the dataset

In [11]:
# Read the sample dataset
import pandas as pd

df = pd.read_csv("dataset/sample_dataset.csv")
print(df.head())

                          title  \
0  Marinated Flank Steak Recipe   
1           French Chicken Stew   
2                Glazed Carrots   
3               Moms Pie Dough    
4      Pretzel Salad Or Dessert   

                                         ingredients  \
0  ["1 1/2 pound flank steak", "1/2 c. finely min...   
1  ["1 tablespoon rosemary", "1 teaspoon thyme", ...   
2  ["3 to 4 carrots", "1 1/2 Tbsp. butter", "1/3 ...   
3  ["4.5 Cups Flour", "1.5 Tsp Salt", "Pinch Baki...   
4  ["2 c. crushed small thin pretzels (sticks)", ...   

                                          directions  \
0  ["Remove tenderloin from steak.", "Score meat....   
1  ["combine all ingredients in slow cooker (6 qu...   
2  ["Cook 3 to 4 carrots; cut crosswise in 1-inch...   
3  ["Mix all dry ingredients in a bowl.", "", "Ad...   
4  ["Mix and press in baking pan, approximately 1...   

                                                link     source  \
0  cookeatshare.com/recipes/marinated-flank-stea

In [30]:
# Combine the string
import json

def combine_string(row):
    raw_ingredients_array = json.loads(row['ingredients'])
    raw_directions_array = json.loads(row['directions'])
    raw_ner_array = json.loads(row['NER'])

    return (
        f"title {row['title']}\n"
        f"{', '.join(raw_ingredients_array)}\n"
        f"{', '.join(raw_directions_array)}\n"
        f"{', '.join(raw_ner_array)}"
    ).lower()

df["combined"] = df.apply(combine_string, axis=1)
print(df["combined"].head())

0    title marinated flank steak recipe\n1 1/2 poun...
1    title french chicken stew\n1 tablespoon rosema...
2    title glazed carrots\n3 to 4 carrots, 1 1/2 tb...
3    title moms pie dough \n4.5 cups flour, 1.5 tsp...
4    title pretzel salad or dessert\n2 c. crushed s...
Name: combined, dtype: object


### Text Pre-processing

In [46]:
# Clean recipe text
import re

def clean_recipe_text(text):
    # 1. Convert to lowercase
    text = text.lower()

    # 2. Remove fractions and numbers
    text = re.sub(r'\d+/\d+', '', text) # Remove fractions
    text = re.sub(r'\d+\.\d+', '', text) # Remove decimals
    text = re.sub(r'\d+', '', text) # Remove numbers

    # 3. Define a list of units to remove
    noise_list = [
        # Units
        'lb', 'lbs', 'pound', 'pounds', 'oz', 'ounce', 'ounces', 
        'gram', 'grams', 'kg', 'kg', 'g', 'ml', 'clove', 'cloves', 'c',
        'cup', 'cups', 'tsp', 'teaspoon', 'teaspoons', 'tbsp', 
        'tablespoon', 'tablespoons', 'can', 'cans', 'package', 'pkgs',
        # Prep Verbs
        'diced', 'minced', 'chopped', 'sliced', 'shredded', 'grated', 'crushed', 
        'mashed', 'pureed', 'ground', 'slivered', 'halved', 'quartered', 'cubed',
        'grnd',
        # States & Descriptors
        'frozen', 'chilled', 'melted', 'fresh', 'dried', 'cooked', 'raw', 
        'large', 'small', 'medium', 'optional', 'divided', 'drained', 'rinsed'
    ]

    # 4. Create a regex pattern from the noise list
    noise_pattern = r'\b(' + '|'.join(noise_list) + r')\b' # \b is a word boundary to avoid partial matches
    text = re.sub(noise_pattern, '', text)

    # 5. Remove special characters
    text = re.sub(r'[^a-z\s]', '', text)

    # 6. Clean up extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

df['cleaned'] = df['combined'].apply(clean_recipe_text)
print(df['cleaned'].head())

0    title marinated flank steak recipe flank steak...
1    title french chicken stew rosemary thyme bay l...
2    title glazed carrots to carrots butter brown s...
3    title moms pie dough flour salt pinch baking p...
4    title pretzel salad or dessert thin pretzels s...
Name: cleaned, dtype: object


In [ ]:
# Remove stopwords
stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    return ' '.join([word for word in text.split() if word not in stop_words])

df['cleaned'] = df['cleaned'].apply(remove_stopwords)
print(df['cleaned'].head())

0    title marinated flank steak recipe flank steak...
1    title french chicken stew rosemary thyme bay l...
2    title glazed carrots carrots butter brown suga...
3    title moms pie dough flour salt pinch baking p...
4    title pretzel salad dessert thin pretzels stic...
Name: cleaned, dtype: object


In [50]:
# Using Snowball Stemmer
stemmer = SnowballStemmer('english')

df['cleaned'] = df['cleaned'].apply(lambda x: ' '.join([stemmer.stem(word) for word in x.split()]))

### Create the Vector Space

In [52]:
# Build the vocabulary
vocabulary = set()
for doc in df['cleaned']:
    for word in doc.split():
        vocabulary.add(word)

print(len(vocabulary))

8115


In [53]:
# Calculate the term frequency matrix
### TF(t, d) = Number of times term t appears in document d / Total number of terms in document d
def calculate_term_frequency_matrix(row):
    all_terms = row.split()
    total_terms = len(all_terms)
    term_freq = {}
    for term in all_terms:
        if term in term_freq:
            term_freq[term] += 1/total_terms
        else:
            term_freq[term] = 1/total_terms
    return term_freq

df['term_frequency'] = df['cleaned'].apply(calculate_term_frequency_matrix)


In [58]:
# Calculate the inverse document frequency
### IDF(t, D) = log(Total number of documents / Number of documents containing term t)
inverse_document_frequency = {}
total_documents = len(df)

for term in vocabulary:
    count = 0
    for row in df['cleaned']:
        if term in row:
            count += 1
    inverse_document_frequency[term] = math.log(total_documents / count)

In [70]:
# Calculate the TD-IDF matrix
### TF-IDF(t, d, D) = TF(t, d) * IDF(t, D)
def calculate_tf_idf_matrix(row):
    tfidf = dict()
    for key, value in row.items():
        tfidf[key] = value * inverse_document_frequency[key]
    return tfidf

df['tfidf'] = df['term_frequency'].apply(calculate_tf_idf_matrix)

### Saving the new data

In [75]:
# Save the dataframe
df.to_csv("5000_data_with_tf_idf.csv", index=False)

In [76]:
# Save vocabulary
pickle.dump(vocabulary, open("vocabulary.pkl", "wb"))